In [1]:
import re
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report
from sklearn.pipeline import Pipeline

NEGATION_WORDS = {"not", "no", "never", "n't", "cannot", "cant", "without"}
DEFAULT_STOP_WORDS = {
    "a", "an", "the", "is", "are", "was", "were", "be", "been", "being",
    "of", "in", "on", "at", "to", "for", "and", "or", "but", "with",
    "this", "that", "these", "those", "it", "its", "as", "so", "very",
    "i", "my", "me", "we", "our", "you", "your",
} - NEGATION_WORDS

def clean_text(text: str) -> str:
    text = text.lower().strip()
    text = re.sub(r"http\S+|www\.\S+", " ", text)
    text = re.sub(r"\S+@\S+", " ", text)
    text = re.sub(r"<[^>]+>", " ", text)
    text = re.sub(r"[^a-z0-9\s']", " ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text

def tokenize(text: str) -> list:
    return text.split()

def remove_stop_words(tokens: list, stop_words: set = None) -> list:
    stop_words = stop_words if stop_words is not None else DEFAULT_STOP_WORDS
    return [t for t in tokens if t not in stop_words]

def preprocess(text: str, remove_stops: bool = True) -> str:
    cleaned = clean_text(text)
    tokens = tokenize(cleaned)
    if remove_stops:
        tokens = remove_stop_words(tokens)
    return " ".join(tokens)

In [2]:
df = pd.read_csv('feedback.csv')
df['category'].value_counts()

,count
category,
performance,8
payment,7
login,5
support,5
ui,5
bug,4
feature_request,4
general,3


In [3]:
def build_single_label_pipeline():
    return Pipeline([
        ("tfidf", TfidfVectorizer(ngram_range=(1, 2))),
        ("classifier", LogisticRegression(max_iter=1000)),
    ])

def train_category_model(texts, categories, test_size=0.2, random_state=42):
    cleaned_texts = [preprocess(t) for t in texts]

    x_train, x_test, y_train, y_test = train_test_split(
        cleaned_texts, categories, test_size=test_size, random_state=random_state
    )

    model = build_single_label_pipeline()
    model.fit(x_train, y_train)

    predictions = model.predict(x_test)
    print("Category model evaluation")
    print("-" * 40)
    print(classification_report(y_test, predictions, zero_division=0))

    return model

def predict_category(model, text):
    cleaned = preprocess(text)
    return model.predict([cleaned])[0]

In [4]:
model = train_category_model(df['feedback'], df['category'])

Category model evaluation
----------------------------------------
              precision    recall  f1-score   support

         bug       0.00      0.00      0.00         1
     general       0.00      0.00      0.00         2
       login       0.00      0.00      0.00         1
     payment       0.00      0.00      0.00         1
 performance       0.14      1.00      0.25         1
     support       1.00      1.00      1.00         1
          ui       0.00      0.00      0.00         2

    accuracy                           0.22         9
   macro avg       0.16      0.29      0.18         9
weighted avg       0.13      0.22      0.14         9



In [5]:
examples = [
    'The application freezes frequently',
    'OTP is not arriving',
    'Please add a dark mode option',
]

for text in examples:
    print(f'{text!r:40} -> {predict_category(model, text)}')

'The application freezes frequently'     -> performance
'OTP is not arriving'                    -> login
'Please add a dark mode option'          -> feature_request


In [6]:
def train_multilabel_category_model(texts, label_lists):
    cleaned_texts = [preprocess(t) for t in texts]

    vectorizer = TfidfVectorizer(ngram_range=(1, 2))
    x = vectorizer.fit_transform(cleaned_texts)

    mlb = MultiLabelBinarizer()
    y = mlb.fit_transform(label_lists)

    classifier = OneVsRestClassifier(LogisticRegression(max_iter=1000))
    classifier.fit(x, y)

    return vectorizer, classifier, mlb

def predict_multilabel_categories(vectorizer, classifier, mlb, text):
    cleaned = preprocess(text)
    x = vectorizer.transform([cleaned])
    prediction = classifier.predict(x)
    return list(mlb.inverse_transform(prediction)[0])

In [7]:
demo_texts = [
    'Payment keeps failing',
    'The application is very slow',
    'I love the new dashboard',
    'Support solved my issue quickly',
    'Login OTP is not arriving',
    'The app is slow and payment fails',
]
demo_labels = [
    ['payment'],
    ['performance'],
    ['ui'],
    ['support'],
    ['login'],
    ['performance', 'payment'],
]

vectorizer, classifier, mlb = train_multilabel_category_model(demo_texts, demo_labels)

In [8]:
multi_example = 'The app is very slow and payment keeps failing'
predicted = predict_multilabel_categories(vectorizer, classifier, mlb, multi_example)
print(f'{multi_example!r} -> {predicted}')

'The app is very slow and payment keeps failing' -> []
